# LLNN Inference on PYNQ-Z2

This notebook loads the LLNN overlay bitstream, writes binarized MNIST input
vectors to the AXI-Lite input registers, and reads the classification result.

## Prerequisites
Upload these files to `/home/xilinx/llnn/` on the PYNQ board:
- `llnn.bit` — bitstream
- `llnn.hwh` — hardware handoff (same base name!)
- `binarized_20x20_MNIST.txt` — test data (10,000 samples × 400 binary digits)
- `mnist_test_labels.txt` — (optional) one label per line, 10,000 lines

### AXI Address Map (axi_lut_ctrl_hard.sv)
| Offset | Description |
|--------|-------------|
| `0x3000–0x3030` | NET_I input registers (13 × 32-bit words = 400 bits) |
| `0x3034` | NET_O output register (lower 4 bits = class 0–9) |

In [4]:
# You can use this cell to check when the bitstream was generated!
with open("/home/xilinx/actual_llnn.bit", "rb") as f:
    header = f.read(256)
# Print human-readable strings from the header
import re
strings = re.findall(b'[\x20-\x7e]{4,}', header)
for s in strings:
    print(s.decode())

5actual_llnn_wrapper;UserID=0XFFFFFFFF;Version=2023.2
7z020clg400
2026/03/06
11:56:12


## 1. Configuration

In [5]:
# ── Paths ──────────────────────────────────────────────────────────────────────
BITSTREAM_PATH = "/home/xilinx/actual_llnn.bit"
MNIST_DATA     = "/home/xilinx/pynq_notebooks/binarized_20x20_MNIST.txt"
LABELS_FILE    = "/home/xilinx/pynq_notebooks/mnist_test_labels.txt"  # optional

# ── AXI constants (must match axi_lut_ctrl_hard.sv) ─────────────────────────
ADDR_INPUT_BASE  = 0x3000   # 13 × 32-bit input words start here
ADDR_OUTPUT      = 0x3034   # 4-bit classification output
NUM_INPUT_WORDS  = 13       # ceil(400 / 32)
NET_INPUTS       = 400      # 20×20 binarized pixels
AXI_ADDR_SPACE   = 0x4000   # 16 KB

## 2. Load the Overlay

In [6]:
from pynq import Overlay, MMIO

ol = Overlay(BITSTREAM_PATH)
print("Overlay loaded!")
print("IP dict keys:", list(ol.ip_dict.keys()))

Overlay loaded!
IP dict keys: ['llnn_wrapper_bd_0', 'processing_system7_0']


In [7]:
# Get MMIO handle to our LLNN AXI controller
# Find the llnn_wrapper IP in the dictionary
ip_name = [k for k in ol.ip_dict.keys() if "llnn" in k.lower()]
if not ip_name:
    # Fallback: use the first non-system IP
    ip_name = [k for k in ol.ip_dict.keys() if "processing_system" not in k]
ip_name = ip_name[0]

base_addr = ol.ip_dict[ip_name]['phys_addr']
print(f"Using IP: {ip_name}")
print(f"Base address: 0x{base_addr:08X}")

mmio = MMIO(base_addr, AXI_ADDR_SPACE)

Using IP: llnn_wrapper_bd_0
Base address: 0x40000000


## 3. Helper Functions

In [8]:
def write_input(mmio, binary_string):
    """
    Write a 400-character binary string to the 13 AXI input registers.
    Bit 0 of the string maps to bit 0 of register 0 (LSB-first packing).
    """
    bits = [int(c) for c in binary_string[:NET_INPUTS]]
    for w in range(NUM_INPUT_WORDS):
        word = 0
        for b in range(32):
            idx = w * 32 + b
            if idx < len(bits):
                word |= bits[idx] << b
        mmio.write(ADDR_INPUT_BASE + w * 4, word)


def read_output(mmio):
    """Read the 4-bit classification result."""
    return mmio.read(ADDR_OUTPUT) & 0xF


def run_inference(mmio, binary_string):
    """Write input + read output in one call."""
    write_input(mmio, binary_string)
    return read_output(mmio)

## 4. Quick Smoke Test
Write all zeros → read output. This verifies AXI communication is working.

In [9]:
# Single read test
import signal, sys
def timeout_handler(signum, frame):
    print("❌ MMIO READ timed out — AXI bus is hanging!")
    sys.exit(1)

# Set 3-second timeout
signal.signal(signal.SIGALRM, timeout_handler)
signal.alarm(3)

mmio = MMIO(base_addr, 0x4000)
# Read the output register (should be safe — it's combinational)
val = mmio.read(0x3034)
signal.alarm(0)  # cancel timeout
print(f"✅ Read 0x3034 = 0x{val:08X} (class = {val & 0xF})")

✅ Read 0x3034 = 0x00000001 (class = 1)


In [10]:
# Single write test
signal.alarm(3)
mmio.write(0x3000, 0x00000000)  # write first input register
signal.alarm(0)
print("✅ Write to 0x3000 succeeded")

✅ Write to 0x3000 succeeded


In [ ]:
# Write all-zero input
write_input(mmio, "0" * NET_INPUTS)
result = read_output(mmio)
print(f"All-zeros input → class {result}")

# Write all-ones input
write_input(mmio, "1" * NET_INPUTS)
result = read_output(mmio)
print(f"All-ones input  → class {result}")

print("\n✅ AXI communication working!" if result <= 9 else "\n❌ Unexpected output")

## 5. Load MNIST Test Data

In [ ]:
import os

# Load binarized samples (10,000 lines × 400 chars each)
with open(MNIST_DATA, "r") as f:
    samples = [line.strip() for line in f.readlines()]

print(f"Loaded {len(samples)} samples")
print(f"Sample length: {len(samples[0])} bits")

# Load labels if available
labels = None
if os.path.exists(LABELS_FILE):
    with open(LABELS_FILE, "r") as f:
        labels = [int(line.strip()) for line in f.readlines()]
    print(f"Loaded {len(labels)} labels")
else:
    print("No labels file found — running inference without accuracy check.")
    print(f"To generate labels, run this on a machine with torchvision:")
    print(f"  python -c \"")
    print(f"  from torchvision import datasets")
    print(f"  ds = datasets.MNIST('data-mnist', train=False, download=True)")
    print(f"  open('mnist_test_labels.txt','w').write('\\n'.join(str(t) for _,t in ds))")
    print(f"  \"")

## 6. Visualize a Sample
Display a 20×20 binarized image as text art.

In [ ]:
def show_sample(binary_string, idx=None, label=None, prediction=None):
    """Display a 20×20 binarized image as text art."""
    title = f"Sample {idx}" if idx is not None else "Sample"
    if label is not None:
        title += f" | Label: {label}"
    if prediction is not None:
        match = "✅" if prediction == label else "❌"
        title += f" | Predicted: {prediction} {match}"
    print(title)
    print("─" * 22)
    for row in range(20):
        line = binary_string[row * 20 : (row + 1) * 20]
        print(" " + line.replace("0", "· ").replace("1", "  "))
    print()

# Show first sample
pred = run_inference(mmio, samples[0])
show_sample(samples[0], idx=0, label=labels[0] if labels else None, prediction=pred)

## 7. Run Batch Inference

In [ ]:
import time

NUM_SAMPLES = min(len(samples), 1000)  # start with 1000, increase as needed

predictions = []
t0 = time.time()

for i in range(NUM_SAMPLES):
    pred = run_inference(mmio, samples[i])
    predictions.append(pred)

elapsed = time.time() - t0
print(f"Ran {NUM_SAMPLES} inferences in {elapsed:.3f} s")
print(f"Throughput: {NUM_SAMPLES / elapsed:.1f} samples/sec")
print(f"Latency:    {elapsed / NUM_SAMPLES * 1000:.2f} ms/sample")

if labels:
    correct = sum(p == labels[i] for i, p in enumerate(predictions))
    accuracy = correct / NUM_SAMPLES * 100
    print(f"\nAccuracy: {correct}/{NUM_SAMPLES} = {accuracy:.2f}%")

## 8. Confusion Matrix (if labels available)

In [ ]:
if labels:
    # Build confusion matrix
    matrix = [[0] * 10 for _ in range(10)]
    for i in range(NUM_SAMPLES):
        true_label = labels[i]
        pred_label = predictions[i]
        if true_label < 10 and pred_label < 10:
            matrix[true_label][pred_label] += 1

    # Print
    print("Confusion Matrix (rows = true, cols = predicted)")
    print("     " + "  ".join(f"{j:3d}" for j in range(10)))
    print("    " + "─" * 45)
    for i in range(10):
        row = "  ".join(f"{matrix[i][j]:3d}" for j in range(10))
        print(f" {i}  │{row}")
else:
    print("Skipping — no labels loaded.")
    print("Prediction distribution:")
    from collections import Counter
    dist = Counter(predictions)
    for cls in sorted(dist.keys()):
        print(f"  Class {cls}: {dist[cls]} ({dist[cls]/NUM_SAMPLES*100:.1f}%)")

## 9. Interactive — Classify Specific Samples

In [ ]:
# Change the index to inspect specific samples
for idx in [0, 1, 2, 3, 4, 100, 500, 999]:
    pred = run_inference(mmio, samples[idx])
    label = labels[idx] if labels else None
    show_sample(samples[idx], idx=idx, label=label, prediction=pred)

## 10. Raw Register Debug
Useful for debugging AXI communication.

In [ ]:
# Read back all input registers
print("Input registers (after last inference):")
for w in range(NUM_INPUT_WORDS):
    val = mmio.read(ADDR_INPUT_BASE + w * 4)
    print(f"  Reg[{w:2d}] @ 0x{ADDR_INPUT_BASE + w*4:04X} = 0x{val:08X}  ({val:032b})")

print(f"\nOutput register @ 0x{ADDR_OUTPUT:04X} = 0x{mmio.read(ADDR_OUTPUT):08X}")